# Explore swissBOUNDARIES3D

Goal: discover actual column names, verify join keys against votation `geoLevelnummer`.

Run `scripts/download_boundaries.py` first.

In [ ]:
from pathlib import Path
import geopandas as gpd
import fiona

GPKG = Path("../data/raw/swissboundaries3d.gpkg")

## Available layers

In [ ]:
layers = fiona.listlayers(GPKG)
for l in layers:
    print(l)

## Columns per layer

In [ ]:
for layer in layers:
    gdf = gpd.read_file(GPKG, layer=layer, rows=1)
    print(f"\n{layer}")
    print("  ", gdf.drop(columns="geometry").columns.tolist())

## Cantons — inspect rows

In [ ]:
LAYER_CANTONS = "tlm_kantonsgebiet"

cantons = gpd.read_file(GPKG, layer=LAYER_CANTONS)
print(f"{len(cantons)} cantons, CRS: {cantons.crs}")
cantons.drop(columns="geometry")

## Districts — inspect rows

In [ ]:
LAYER_DISTRICTS = "tlm_bezirksgebiet"

districts = gpd.read_file(GPKG, layer=LAYER_DISTRICTS)
print(f"{len(districts)} districts")
districts.drop(columns="geometry").head(20)

## Municipalities — inspect rows

In [ ]:
LAYER_MUNI = "tlm_hoheitsgebiet"

municipalities = gpd.read_file(GPKG, layer=LAYER_MUNI)
print(f"{len(municipalities)} municipalities")
municipalities.drop(columns="geometry").head(20)

## Join key check

Votation data uses:
- `kantone[].geoLevelnummer` — Zürich = `"1"`
- `bezirke[].geoLevelnummer` — Bezirk Affoltern = `"101"`
- `gemeinden[].geoLevelnummer` — Aeugst am Albis = `"1"`, Affoltern am Albis = `"2"`

Find which column in each layer matches those values.

In [ ]:
COL_CANTON_NUM = "kantonsnummer"
COL_DISTRICT_NUM = "bezirksnummer"
COL_MUNI_NUM = "bfs_nummer"

print("Canton 1 should be Zürich:")
print(cantons[cantons[COL_CANTON_NUM] == 1][[COL_CANTON_NUM, "name"]].to_string())

print("\nDistrict 101 should be Bezirk Affoltern:")
print(districts[districts[COL_DISTRICT_NUM] == 101][[COL_DISTRICT_NUM, "name"]].to_string())

print("\nMunicipality 2 should be Affoltern am Albis (if BFS matches votation):")
print(municipalities[municipalities[COL_MUNI_NUM] == 2][[COL_MUNI_NUM, "name"]].to_string())

## Quick plot

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
cantons.to_crs("EPSG:4326").plot(ax=axes[0], edgecolor="black", linewidth=0.5)
axes[0].set_title("Cantons")
districts.to_crs("EPSG:4326").plot(ax=axes[1], edgecolor="black", linewidth=0.3)
axes[1].set_title("Districts")
municipalities.to_crs("EPSG:4326").plot(ax=axes[2], edgecolor="black", linewidth=0.1)
axes[2].set_title("Municipalities")
plt.tight_layout()
plt.show()